# Demo

> To run the code in this notebook, one needs to have the `sent_logic` library installed using `pip`. See the instructions in `README.md`.

The present notebook shows how the `sent_logic` library can be used. The most interesting functionality that the library exports is a "CDCL-based SAT solver" in `sent_logic.sat.solver_cdcl`. Accordingly, the notebook focuses on the following questions:

- What is a SAT solver? What problem is it meant to solve? In other words: What is the SAT problem?
- How can I use the SAT solvers exported by the library?

## Sentential logic

The language of sentential logic consists of _atomic sentences_ (or _atoms_) and _connectives_. Atoms can be either _named variables_ (written `p`, `q`, `my_variable` etc.) or _indexed variables_ (written `#1`, `#2`, etc.). The connectives are:

- negation: `~` – `Not` – "not"
- conjunction: `&` – `And` – "and"
- exclusive disjunction: `^` – `Xor` – "xor"
- inclusive disjunction: `|` – `Or` – "or"
- conditional: `=>` – `Cond` – "if, then"

__Examples:__

In [1]:
from sent_logic import *

# Indexed variables:

var_1                  = parse("#1")
var_2                  = parse("#2")
var_3                  = parse("#3")

# In Python notation:

assert var_1           == Atom(IVar(1))
assert var_2           == Atom(IVar(2))
assert var_3           == Atom(IVar(3))

In [2]:
# Named variables

p                      = parse("p")
q                      = parse("q")
r                      = parse("r")
some_named_var         = parse("some_named_var")

# In Python notation:

assert p               == Atom(NVar("p"))
assert q               == Atom(NVar("q"))
assert r               == Atom(NVar("r"))

assert some_named_var  == Atom(NVar("some_named_var"))

In [3]:
# Compound sentences:

lem                    = parse("p | ~p")
lem_xor                = parse("p ^ ~p")

assert lem             == Or(p, Not(p))
assert lem_xor         == Xor(p, Not(p))

not_inv                = parse("~~p => p")

assert not_inv         == Cond(Not(Not(p)), p)

and_comm               = parse("p & q => q & p")
and_assoc              = parse("p & (q & r) => (p & q) & r")
and_curry              = parse("(p & q => r) => (p => q => r)")

assert and_comm        == Cond(And(p, q), And(q, p))
assert and_assoc       == Cond(And(p, And(q, r)), And(And(p, q), r))
assert and_curry       == Cond(Cond(And(p, q), r), Cond(p, Cond(q, r)))

or_comm                = parse("p | q => q | p")
or_assoc               = parse("p | (q | r) => (p | q) | r")
or_curry               = parse("(p | q => r) => (p => r) & (q => r)")

assert or_comm         == Cond(Or(p, q), Or(q, p))
assert or_assoc        == Cond(Or(p, Or(q, r)), Or(Or(p, q), r))
assert or_curry        == Cond(Cond(Or(p, q), r), And(Cond(p, r), Cond(q, r)))

distrib_1              = parse("p & (q | r) => p & q | p & r")
distrib_2              = parse("p | q & r => (p | q) & (p | r)")
de_morgan_1            = parse("~(p & q) => ~p | ~q")
de_morgan_2            = parse("~(p | q) => ~p & ~q")

assert distrib_1       == Cond(And(p, Or(q, r)), Or(And(p, q), And(p, r)))
assert distrib_2       == Cond(Or(p, And(q, r)), And(Or(p, q), Or(p, r)))
assert de_morgan_1     == Cond(Not(And(p, q)), Or(Not(p), Not(q)))
assert de_morgan_2     == Cond(Not(Or(p, q)), And(Not(p), Not(q)))

de_morgan_1_xor        = parse("(p & q) ^ (~p | ~q)")
de_morgan_2_xor        = parse("(p | q) ^ (~p & ~q)")

assert de_morgan_1_xor == Xor(And(p, q), Or(Not(p), Not(q)))
assert de_morgan_2_xor == Xor(Or(p, q), And(Not(p), Not(q)))

The `parse` function is used to convert a sentence written in the notation of sentential logic into a Python representation of that sentence. Atoms are constructed using `Atom`. Indexed variables _taken as atomic sentences_ are represented as `Atom(IVar(i))` rather than `Atom(i)` or `IVar(i)` (the same holds for named variables). The `display` function can be used to convert a Python representation of a sentence into the notation of sentential logic. Note that `display` omits redundant parentheses meaning that, e.g., `display(parse("p => (q => r)"))` evaluates to `"p => q => r"`. (The `str` function is equivalent to `display` on sentences which has the convenient consequence that sentences can be constructed using f-string syntax.)

__Examples:__

In [4]:
assert display(lem)             == "p | ~p"
assert display(lem_xor)         == "p ^ ~p"

assert display(not_inv)         == "~~p => p"

assert display(and_comm)        == "p & q => q & p"
assert display(and_assoc)       == "p & q & r => (p & q) & r"
assert display(and_curry)       == "(p & q => r) => p => q => r"

assert display(or_comm)         == "p | q => q | p"
assert display(or_assoc)        == "p | q | r => (p | q) | r"
assert display(or_curry)        == "(p | q => r) => (p => r) & (q => r)"

assert display(distrib_1)       == "p & (q | r) => p & q | p & r"
assert display(distrib_2)       == "p | q & r => (p | q) & (p | r)"
assert display(de_morgan_1)     == "~(p & q) => ~p | ~q"
assert display(de_morgan_2)     == "~(p | q) => ~p & ~q"

assert display(de_morgan_1_xor) == "p & q ^ (~p | ~q)"
assert display(de_morgan_2_xor) == "(p | q) ^ ~p & ~q"

Using f-string syntax:

In [5]:
p_and_q         = parse("p & q")
q_and_p         = parse("q & p")

assert and_comm == parse(f"{p_and_q} => {q_and_p}")

### Semantics

A valuation is an assignment of boolean values to variables. In the library, valuations are represented by `Valuation` objects; that is, `dict` objects with keys in `str | int` and values in `bool`. The `eval_sent` function can be used to evaluate a sentence against a valuation that contains every variable appearing in that sentence. If a sentence `sent` evaluates to `True` under valuation `vln`, then `vln` is said to _satisfy_ `sent`. A sentence is _satisfiable_ if some valuation satisfies it.

__Example:__

We can get the `set` of variables (that is, variable names and indices) appearing in a sentence using `variables` and then generate all valuations on those variables using `valuations`. The following examples uses these functions together with the `tabulate` library (`pip install tabulate`) to generate truth tables.

In [6]:
from tabulate import tabulate

def truth_table(*sents):
    """
    Returns a truth-table for a list of sentences.
    """
    vars = variables(*sents)
    ordered_vars = list(vars)
    header = ordered_vars + [display(sent) for sent in sents]
    table = [header]
    for vln in valuations(vars):
        table.append(
            [vln[v] for v in ordered_vars] +
            [eval_sent(sent, vln) for sent in sents]
        )
    return tabulate(table, tablefmt="html")

In [7]:
truth_table(parse("~p"))

p,~p
False,True
True,False


In [8]:
truth_table(parse("p & q"), parse("p ^ q"), parse("p | q"), parse("p => q"))

q,p,p & q,p ^ q,p | q,p => q
False,False,False,False,False,True
False,True,False,True,True,False
True,False,False,True,True,True
True,True,True,False,True,True


A sentence is a _validity_ (or a _tautology_) if every valuation satisfies it. `check_valid` can be used to check whether a sentence is a validity. Similarly, `check_sat` can be used to check whether a sentence is satisfiable. These functions iterate through all valuations as in the example above — the number of valuation is exponential in the number of variables which means that this strategy is infeasible for large sentences.

## The SAT problem

SAT is the following decision problem: For a CNF sentence, decide whether that sentence is satisfiable. A decision proceduce for this problem is called a SAT solver. When the sentence is satisfiable, the procedure can return a valuation witnessing this. (Some SAT solvers can also return a proof of unsatisfiability when the sentence is unsatisfiable — the solvers from this library do not have this feature, however.)

A literal is a sentence of the form `v` or `~v` where `v` is a variable. A clause is a sentence of the form `l_1 | l_2 | ... | l_n` where the `l_i` are literals. Note that sentences of the form `a_1 & a_2 & ... & a_n => b_1 | b_2 | ... | b_n` where the `a_i` and `b_j` are literals are equivalent to clauses. A CNF is a sentence of the form `c_1 & c_2 & ... & c_n` where `c_i` are clauses. In the library, CNF sentences have a special representation distinct from the general representation introduced above.

__The special representation allows indexed variables only.__ Literals `#1`, `~#1`, `#2`, `~#2`, etc. are coded as non-zero integers `1`, `-1`, `2`, `-2`, etc. Clauses are represented as lists of literals (that is, lists of non-zero integers) and CNF sentences themselves as lists of clauses (that is, lists of lists of non-zero integers). The empty clause always evaluates to false; the empty CNF always evaluates to true. The library exports type aliases `Var`, `Lit`, `Clause`, `Clauses` for variables (positive integers), literals (integers), clauses (lists of integers), and CNF sentences (lists of lists of integers) respectively.

In [9]:
from sent_logic.sat import *

def cnf_truth_table(*cnfs: Clauses):
    """
    Returns a truth-table for a CNF.
    """
    # We can use `nvars` to get the maximum variable in a CNF.
    n = nvars(*cnfs)
    vars = list(range(1, n + 1))
    header = vars + [repr(cnf) for cnf in cnfs]
    table = [header]
    # `valuations_` is used to enumerate valuations of indexed variables
    # (given the maximum variable).
    for vln in valuations_(n):
        # `eval_clauses` evaluates clauses under valuations.
        table.append(
            [vln[v] for v in vars] +
            [eval_cnf(cnf, vln) for cnf in cnfs]
        )
    return tabulate(table, tablefmt="html")

In [10]:
# `(#1 => #2) & (#2 => #3)` in clauses.
cnf_truth_table([[-1, 2]], [[-2, 3]], [[-1, 2], [-2, 3]])

1,2,3,"[[-1, 2]]","[[-2, 3]]","[[-1, 2], [-2, 3]]"
False,False,False,True,True,True
False,False,True,True,True,True
False,True,False,True,False,False
False,True,True,True,True,True
True,False,False,False,True,False
True,False,True,False,True,False
True,True,False,True,False,False
True,True,True,True,True,True


Sentences can be converted into equivalent CNF sentences. This conversion is provided by the `into_cnf` function. This function can also be used to convert a CNF in the general representation into a CNF in the special representation. Note that in the general case, the size of the resulting CNFs can be exponential in the size of the original sentence.

In [11]:
assert into_cnf(parse("#1 & ~#2"))                == [[1], [-2]]
assert into_cnf(parse("#1 | ~#2"))                == [[1, -2]]
assert into_cnf(parse("#1 => #2"))                == [[-1, 2]]
assert into_cnf(parse("(#1 => #2) & (#2 => #3)")) == [[-1, 2], [-2, 3]]
assert into_cnf(parse("#1 & #2 => #3 | ~#4"))     == [[-1, -2, 3, -4]]
assert into_cnf(parse("#1 ^ #2"))                 == [[1, 2], [-2, -1]]

In [52]:
def into_cnf_sound(sent: ISent) -> bool:
    n = nindices(sent)
    cnf = into_cnf(sent)
    return all(
        eval_sent(sent, vln) == eval_cnf(cnf, vln)
        for vln in valuations_(n)
    )

assert into_cnf_sound(parse("#1 & ~#2"))
assert into_cnf_sound(parse("#1 | ~#2"))
assert into_cnf_sound(parse("#1 => #2"))
assert into_cnf_sound(parse("(#1 => #2) & (#2 => #3)"))
assert into_cnf_sound(parse("#1 & #2 => #3 | ~#4"))
assert into_cnf_sound(parse("#1 ^ #2"))
assert into_cnf_sound(parse("(#1 & #2) | (#3 & #4) | (#5 & #6)"))

In [53]:
# Exponential blow-up:
# Sentence has size ~ 2n, while the CNF has size ~ 2^n.
into_cnf(parse("(#1 & #2) | (#3 & #4) | (#5 & #6)"))

[[1, 3, 5],
 [1, 3, 6],
 [1, 4, 5],
 [1, 4, 6],
 [2, 3, 5],
 [2, 3, 6],
 [2, 4, 5],
 [2, 4, 6]]

The solvers live in the `sent_logic.sat.solver_dpll` and `sent_logic.sat.solver_cdcl` modules as functions named `solve`. The interface of both functions is the same. The input is a list of clauses (that is, a CNF sentence) and the output is an object of type `SolverResult`. The result objects has a field `sat` indicating whether the clauses are satisfiable. It also has a field `vln` which contains a satisfying valuation when the clauses are satisfiable and `None` otherwise.

In [54]:
from sent_logic.sat.solver_cdcl import solve

cnf = into_cnf(parse("(#1 => #2) & ~(~#1 => ~#2)"))
solve(cnf)

SolverResult(sat=True, vln={1: False, 2: True})

### Checking satisfiability for sentences



Converting a sentences into an _equivalent_ CNF and solving the clauses of _that_ CNF is generally not feasible. This is because, as expained above, the size of an equivalent CNF can be exponential in the size of the original sentence. Instead, a sentence can be converted into an _equisatisfiable_ CNF. More precisely, we start with a sentence `sent` having variables `vars` and generate a CNF `sent_cnf` with variables `vars + aux_vars` such that (a) every valuation on `vars` satisfying `sent` can be extended into a valuation on `vars + aux_vars` satisfying `sent_cnf`; and (b) every valuation `vln` on `vars + aux_vars` that satisfies `sent_cnf` also satisfies `sent`. It is possible to convert a sentence into an equisatisfiable CNF so that the size of the CNF is linear in the size of the sentence. This conversion is provided by the `into_equisat_cnf` function.

We illustrate how it can be used to check the satisfiability of large sentences. We generate these large sentences randomly:

In [50]:
import random

def random_sent(nvars: int, size: int) -> ISent:
    """
    Generates a random sentence of size ~ `size` with variables `nvars`
    """
    assert size >= 0
    if size == 0:
        return Atom(IVar(random.randint(1, nvars)))
    
    match random.randint(1, 5):
        case 1: return Not(random_sent(nvars, size - 1))
        case 2: return And(random_sent(nvars, size // 2), random_sent(nvars, size // 2))
        case 3: return Xor(random_sent(nvars, size // 2), random_sent(nvars, size // 2))
        case 4: return Or(random_sent(nvars, size // 2), random_sent(nvars, size // 2))
        case 5: return Cond(random_sent(nvars, size // 2), random_sent(nvars, size // 2))

__Example:__

In [49]:
rand_sent = random_sent(500, 10_000)
rand_cnf = into_equisat_cnf(rand_sent)
result = solve(rand_cnf)

if not result.sat:
    print("UNSAT")
else:
    assert result.vln is not None
    # Verify:
    assert eval_sent(rand_sent, result.vln)
    print("SAT")

SAT


Most randomly generated sentences are satisfiable (which is t)

In [38]:
total_sents = 10_000
sat_sents = 0

for _ in range(total_sents):
    rand_sent = random_sent(50, 50)
    rand_cnf  = into_equisat_cnf(rand_sent)
    result    = solve(rand_cnf)
    
    if result.sat:
        sat_sents += 1
        assert result.vln is not None
        assert eval_sent(rand_sent, result.vln)

print(f"%SAT: {sat_sents / total_sents * 100:.2f}")

%SAT: 99.78


### Solving Sudoku

We illustrate the SAT solver by using it solve Sudoku problems: There is a simple method for reducing a Sudoku instance into a SAT instance. This means that programming a Sudoku solver using the library is relatively easy. First, we define a representations for the Sudoku problems themselves:

In [16]:
from collections.abc import Iterable
from itertools import combinations

from sent_logic.sat.solver_cdcl import solve

type Grid = list[list[int]]
"""
A Sudoku grid: A 9x9 matrix with integers 0–9 as entries. (Empty cells are coded by zeros.)
"""

def sudoku_table(grid: Grid):
    return tabulate(
        [[(" " if n == 0 else n) for n in row] for row in grid],
        tablefmt="html",
    )

We declare the Sudoku problem we want to solve.

In [17]:
SUDOKU: Grid = [
    [9, 2, 0, 3, 0, 6, 0, 0, 0],
    [0, 0, 1, 0, 2, 4, 6, 0, 0],
    [5, 0, 0, 0, 0, 0, 0, 0, 1],
    [0, 4, 0, 0, 0, 7, 0, 0, 0],
    [1, 0, 3, 4, 0, 2, 7, 0, 6],
    [0, 0, 0, 1, 0, 0, 0, 8, 0],
    [8, 0, 0, 0, 0, 0, 0, 0, 2],
    [0, 0, 7, 2, 8, 0, 9, 0, 0],
    [0, 0, 0, 6, 0, 1, 0, 3, 7],
]

sudoku_table(SUDOKU)

9,2,,3,,6,,,
,,1,,2,4,6,,
5,,,,,,,,1
,4,,,,7,,,
1,,3,4,,2,7,,6
,,,1,,,,8,
8,,,,,,,,2
,,7,2,8,,9,,
,,,6,,1,,3,7


We use atomic sentences of the form `Cell (i, j) contains n` to encode statements about the Sudoku grid into clauses. We use the obvious unit clauses to encode the initial state of the problem. Note that SAT instance generated from a Sudoku puzzles will have `9^3` variables. The search space is therefore of size `2^(9^3)` — solving such SAT instances using the brute force method is not feasible. The solvers exported by the library will solve these instances in an instant.

In [18]:
def encode_assignment(i: int, j: int, n: int) -> Var:
    """
    Encodes tuples `(i, j, n)` (where `i` and `j` are integers 0–8 and `n`
    an integer 1–9) into variable indices. An assignment of `True` to the
    corresponding variable should signify that the cell at `(i, j)` contains `n`.
    """
    assert 0 <= i < 9 and 0 <= j < 9 and 1 <= n <= 9
    part_i = i
    part_j = 9 * j
    part_n = 9 ** 2 * (n - 1)
    return 1 + part_i + part_j + part_n


def encode_sudoku(grid: Grid) -> Clauses:
    """
    Encode a Sudoku problem into unit clauses.
    """
    return [
        # A unit clause:
        [encode_assignment(i, j, grid[i][j])]
        for i in range(9)
        for j in range(9)
        if grid[i][j] != 0
    ]

def decode_assignment(v: Var) -> tuple[int, int, int]:
    """
    The inverse to `encode_assignment`.
    """
    part_i = v - 1
    part_j = part_i // 9
    part_n = part_j // 9
    return part_i % 9, part_j % 9, 1 + part_n % 9

def decode_sudoku(vln: IValuation) -> Grid:
    """
    Decodes a valuation for indexed variables into a Sudoku grid.
    """
    grid = [[0 for _ in range(9)] for _ in range(9)]

    for var, val in vln.items():
        if val:
            i, j, n = decode_assignment(var)
            assert grid[i][j] == 0
            grid[i][j] = n

    return grid

Next, we encode the general rules of Sudoku in terms of the chosen atomic sentences:

- Every cell contains a unique number: for each `i` and `j`, there is a unique `n` such that `Cell (i, j) contains n`.
- Every row contains every number once: for each `i` and `n` there is a unique `j` such that `Cell (i, j) contains n`.
- Every column contain every number once: For each `j` and `n` there is a unique `i` such that `Cell (i, j) contains n`.
- Every 3x3 region contains every number once (harder to transcribe).

In [19]:
def alternatives(vs: Iterable[Var]) -> Clauses:
    """
    Encodes the exclusive disjunction of the `vs` into clauses.
    """
    vs = list(vs)
    # The first clauses says that one of the v_i must hold.
    # The other clauses say that v_i and v_j cannot hold simultaneously for i != j.
    return [vs] + [[-v1, -v2] for v1, v2 in combinations(vs, 2)]


def encode_sudoku_rules() -> Clauses:
    """
    Encodes the general rules of Sudoku into clauses.
    """
    clauses = []

    # Every cell should contain one number:
    for i in range(9):
        for j in range(9):
            clauses += alternatives(encode_assignment(i, j, n) for n in range(1, 10))

    # Every row and column should contain every number:
    for i in range(9):
        for n in range(1, 10):
            clauses += alternatives(encode_assignment(i, j, n) for j in range(9))
            clauses += alternatives(encode_assignment(j, i, n) for j in range(9))

    # Every 3x3 should contain every number:
    for i in range(3):
        for j in range(3):
            for n in range(1, 10):
                clauses += alternatives(
                    encode_assignment(i1 + 3 * i, j1 + 3 * j, n)
                    for i1 in range(3)
                    for j1 in range(3)
                )

    return clauses


def solve_sudoku(grid: Grid) -> Grid | None:
    """
    Solves a Sudoku puzzle using a SAT solver.
    """
    clauses = encode_sudoku_rules() + encode_sudoku(grid)
    result = solve(clauses)

    if not result.sat:
        return None
    assert result.vln is not None
    return decode_sudoku(result.vln)

In [20]:
solved_sudoku = solve_sudoku(SUDOKU)
sudoku_table(solved_sudoku)

9,2,8,3,1,6,5,7,4
3,7,1,5,2,4,6,9,8
5,6,4,9,7,8,3,2,1
6,4,5,8,3,7,2,1,9
1,8,3,4,9,2,7,5,6
7,9,2,1,6,5,4,8,3
8,3,6,7,5,9,1,4,2
4,1,7,2,8,3,9,6,5
2,5,9,6,4,1,8,3,7
